In [205]:
import os
import numpy as np
import pandas as pd

data_dir = r"C:\Users\psiqg\.virtual_documents\data_visualization\Data Visualization\Data-visualisation\Data-visualisation\data\raw\openkbp\provided-data\train-pats"

print("Pacientes encontrados:", len(os.listdir(data_dir)))


Pacientes encontrados: 204


In [206]:
results = []

for patient in os.listdir(data_dir):

    patient_path = os.path.join(data_dir, patient)

    if not os.path.isdir(patient_path):
        continue

    for file in os.listdir(patient_path):

        if "dose.csv" in file:

            dose_path = os.path.join(patient_path, file)

            dose = pd.read_csv(dose_path)

            mean_dose = dose.values.mean()
            max_dose = dose.values.max()

            results.append({
                "patient": patient,
                "mean_dose": mean_dose,
                "max_dose": max_dose
            })

In [208]:
df = pd.DataFrame(results)

df.to_csv(r"C:\Users\psiqg\.virtual_documents\data_visualization\Data Visualization\Data-visualisation\Data-visualisation\data\processed\dose_summary.csv", index=False)

df.head()

,patient,mean_dose,max_dose
0,pt_1,533321.571418,1499941.0
1,pt_10,570468.478549,1650903.0
2,pt_100,557034.777639,1487599.0
3,pt_101,551703.925667,1487457.0
4,pt_102,561889.098352,1651824.0


In [178]:
df["mean_dose"] = df["mean_dose"] / 10000
df["max_dose"] = df["max_dose"] / 10000

In [213]:
df.to_csv(r"C:\Users\psiqg\.virtual_documents\data_visualization\Data Visualization\Data-visualisation\Data-visualisation\data\processed\dose_summary.csv", index=False)

In [212]:
import os
os.listdir(r"C:\Users\psiqg\.virtual_documents\data_visualization\Data Visualization\Data-visualisation\Data-visualisation\data\processed")

['dose_summary.csv', 'openkbp_structure_doses.csv']

In [182]:
# extremos
max_point = df.loc[df["max_dose"].idxmax()]
min_point = df.loc[df["max_dose"].idxmin()]

df["tipo"] = "normal"
df.loc[df.index == max_point.name, "tipo"] = "maximo"
df.loc[df.index == min_point.name, "tipo"] = "minimo"

# nombres para la leyenda
df["tipo"] = df["tipo"].replace({
    "normal": "Normal",
    "maximo": "Máximo",
    "minimo": "Mínimo"
})

base = alt.Chart(df).encode(
    x=alt.X(
        "mean_dose:Q",
        title="Dosis media (cGy)",
        scale=alt.Scale(domain=[52, 60])
    ),
    y=alt.Y(
        "max_dose:Q",
        title="Dosis máxima (cGy)",
        scale=alt.Scale(domain=[145,170])
    )
)

# puntos
points = base.mark_circle().encode(
    size=alt.condition(
        alt.datum.tipo != "Normal",   # extremos más grandes
        alt.value(220),
        alt.value(80)
    ),
    color=alt.Color(
        "tipo:N",
        title="Dato",
        scale=alt.Scale(
            domain=["Normal", "Máximo", "Mínimo"],
            range=["steelblue", "red", "green"]
        )
    ),
    tooltip=[
        alt.Tooltip("patient:N", title="Paciente"),
        alt.Tooltip("mean_dose:Q", title="Dosis media (cGy)", format=".2f"),
        alt.Tooltip("max_dose:Q", title="Dosis máxima (cGy)", format=".2f")
    ]
)

# intervalo de confianza
confidence = base.transform_regression(
    "mean_dose", "max_dose", extent=[52, 60]
).mark_area(
    opacity=0.18,
    color="gray"
)

# línea de tendencia
trend = base.transform_regression(
    "mean_dose", "max_dose", extent=[52, 60]
).mark_line(
    color="black",
    size=3
)

# etiquetas para extremos
labels = alt.Chart(df[df["tipo"] != "Normal"]).mark_text(
    dx=7,
    dy=-7,
    fontSize=12,
    fontWeight="bold"
).encode(
    x="mean_dose:Q",
    y="max_dose:Q",
    text="patient:N"
)

chart = alt.layer(confidence, points, trend, labels).properties(
    title="Relación entre dosis media y dosis máxima por paciente",
    width=650,
    height=450
).interactive()

chart

alt.LayerChart(...)

In [183]:
import os
import pandas as pd
import numpy as np

ruta_base = r"C:\Users\psiqg\.virtual_documents\data_visualization\Data Visualization\Data-visualisation\Data-visualisation\data\raw\openkbp\provided-data\test-pats"

estructuras = [
    "Brainstem",
    "SpinalCord",
    "LeftParotid",
    "RightParotid",
    "PTV56",
    "PTV63",
    "PTV70"
]

N_VOXELS = 128 * 128 * 128

def cargar_sparse_csv(path, n_voxels=N_VOXELS, default_value=0.0):
    """
    Lee un CSV sparse de OpenKBP.
    Devuelve un vector plano de longitud n_voxels.
    """
    df = pd.read_csv(path, index_col=0)

    arr = np.full(n_voxels, default_value, dtype=float)

    # Si existe columna 'data', usarla
    if "data" in df.columns:
        indices = df.index.astype(int).to_numpy()
        valores = df["data"].fillna(1).astype(float).to_numpy()
        arr[indices] = valores
    else:
        # por si el csv solo trae índices
        indices = df.index.astype(int).to_numpy()
        arr[indices] = 1.0

    return arr

resultados = []

for paciente in os.listdir(ruta_base):
    carpeta_paciente = os.path.join(ruta_base, paciente)

    if not os.path.isdir(carpeta_paciente):
        continue

    dose_path = os.path.join(carpeta_paciente, "dose.csv")
    if not os.path.exists(dose_path):
        continue

    dose = cargar_sparse_csv(dose_path, default_value=0.0)

    for estructura in estructuras:
        estructura_path = os.path.join(carpeta_paciente, f"{estructura}.csv")
        if not os.path.exists(estructura_path):
            continue

        mask = cargar_sparse_csv(estructura_path, default_value=0.0)

        voxels = dose[mask > 0]

        if len(voxels) == 0:
            continue

        resultados.append({
            "patient": paciente,
            "structure": estructura,
            "mean_dose": float(np.mean(voxels)),
            "max_dose": float(np.max(voxels))
        })

df_total = pd.DataFrame(resultados)

print(df_total.head())
print(df_total["structure"].unique())

  patient     structure  mean_dose  max_dose
0  pt_241     Brainstem   3.661239    25.675
1  pt_241    SpinalCord  19.096967    34.281
2  pt_241   LeftParotid  27.276652    59.921
3  pt_241  RightParotid  25.205656    57.650
4  pt_241         PTV56  59.478529    65.504
<StringArray>
[   'Brainstem',   'SpinalCord',  'LeftParotid', 'RightParotid',
        'PTV56',        'PTV63',        'PTV70']
Length: 7, dtype: str


In [184]:
print(df_total.isna().sum())

patient      0
structure    0
mean_dose    0
max_dose     0
dtype: int64


In [185]:
df_total = df_total.dropna()

In [186]:
df_total = df_total.dropna(subset=["mean_dose", "max_dose"])

In [187]:
df_total["mean_dose"] = pd.to_numeric(df_total["mean_dose"], errors="coerce")
df_total["max_dose"] = pd.to_numeric(df_total["max_dose"], errors="coerce")

In [188]:
df_total = df_total[df_total["mean_dose"] >= 0]

In [189]:
print(df_total.describe())

        mean_dose    max_dose
count  607.000000  607.000000
mean    39.430358   59.050962
std     23.904836   18.010520
min      0.013746    0.164000
25%     18.466570   41.763500
50%     36.267778   70.000000
75%     61.457235   72.985000
max     73.940502   79.977000


In [190]:
print(df_total["structure"].value_counts())

structure
PTV70           100
RightParotid     99
LeftParotid      98
PTV56            91
Brainstem        89
SpinalCord       88
PTV63            42
Name: count, dtype: int64


In [191]:
orden = [
    "PTV70",
    "PTV63",
    "PTV56",
    "LeftParotid",
    "RightParotid",
    "Brainstem",
    "SpinalCord"
]

In [194]:
ruta_base = "data/raw/openkbp/provided-data/test-pats"

In [197]:
ruta_base = r"C:\Users\psiqg\.virtual_documents\data_visualization\Data Visualization\Data-visualisation\Data-visualisation\data\raw\openkbp\provided-data\test-pats"

In [198]:
print("Ruta usada:", ruta_base)
print("Existe:", os.path.exists(ruta_base))

Ruta usada: C:\Users\psiqg\.virtual_documents\data_visualization\Data Visualization\Data-visualisation\Data-visualisation\data\raw\openkbp\provided-data\test-pats
Existe: True


In [199]:
df_plot = df_total.copy()
df_plot["mean_dose"] = pd.to_numeric(df_plot["mean_dose"], errors="coerce")
df_plot["max_dose"] = pd.to_numeric(df_plot["max_dose"], errors="coerce")
df_plot = df_plot.dropna(subset=["patient", "structure", "mean_dose", "max_dose"])

orden = [
    "PTV70", "PTV63", "PTV56",
    "LeftParotid", "RightParotid",
    "Brainstem", "SpinalCord"
]

selector = alt.selection_point(
    name="selector_estructura_openkbp",
    fields=["structure"],
    bind="legend"
)

chart = alt.Chart(df_plot).mark_circle(size=70).encode(
    x=alt.X(
        "structure:N",
        sort=orden,
        title="Estructura anatómica",
        axis=alt.Axis(labelAngle=-30)
    ),
    y=alt.Y(
        "mean_dose:Q",
        title="Dosis media (Gy)"
    ),
    color=alt.Color(
        "structure:N",
        title="Estructura anatómica",
        sort=orden
    ),
    opacity=alt.condition(selector, alt.value(1), alt.value(0.15)),
    tooltip=[
        alt.Tooltip("patient:N", title="Paciente"),
        alt.Tooltip("structure:N", title="Estructura"),
        alt.Tooltip("mean_dose:Q", title="Dosis media", format=".2f"),
        alt.Tooltip("max_dose:Q", title="Dosis máxima", format=".2f")
    ]
).add_params(
    selector
).properties(
    title="Dosis media por estructura anatómica",
    width=750,
    height=450
)

chart

alt.Chart(...)